# Advanced EDA: Part-to-Whole Analysis (Revenue Proportions & Share)

## Executive Summary & Analytical Domain
This notebook performs **Part-to-Whole Proportional Analysis** to evaluate the relative revenue contribution of product categories and geographic regions. Determining the share of total revenue is essential for:
- Evaluating enterprise dependence on individual product lines (Pareto Principle / 80-20 rule).
- Identifying core revenue pillars versus secondary or emerging categories.
- Benchmarking regional market penetration and identifying expansion opportunities.

## Key Performance Indicators (KPIs) & Methodology
- **Total Sales (`total_sales`)**: Aggregate revenue within a specific segment.
- **Overall Enterprise Sales (`overall_sales`)**: Global sales baseline across all segments computed via unbounded window function `SUM(total_sales) OVER ()`.
- **Percentage of Total (`percentage_of_total`)**: Formatted proportional contribution:
  $$\text{percentage\_of\_total} = \text{ROUND}\left(\frac{\text{total\_sales}}{\text{overall\_sales}} \times 100, 2\right)\%$$

## Dialect Enhancements & Bug Fixes
- **Integer Truncation Protection**: Uses `100.0 * total_sales / SUM(total_sales) OVER ()` to enforce floating-point arithmetic.
- **Sorting Parity**: In Query 2 (Regional Sales), sorting is performed on the numeric underlying metric `ORDER BY total_sales DESC` rather than the alphanumeric formatted percentage string to guarantee mathematical descending order.

## Data Contracts & Layer Dependencies
- **Target Layer**: `gold`
- **Tables Referenced**:
  - `gold.fact_sales` (`sales_amount`, `product_key`, `customer_key`)
  - `gold.dim_products` (`product_key`, `category`)
  - `gold.dim_customers` (`customer_key`, `country`)

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Setup display helper and session verification
try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure gold database context
spark.sql("USE gold")
print("Session configured. Active database context: gold")

## 1. Product Category Revenue Contribution

### Business Context & Objective
Analyzes what portion of global sales is generated by each major product category (e.g., Bikes, Components, Clothing, Accessories).

In [ ]:
# COMMAND ----------
# 1. Product Category Sales Share
df_category_share = spark.sql("""
WITH category_sales AS (
    SELECT 
        p.category,
        sum(f.sales_amount) AS total_sales
    FROM gold.fact_sales f
    LEFT JOIN gold.dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.category
)
SELECT 
    category, 
    total_sales,
    sum(total_sales) OVER () AS overall_sales,
    concat(round(100.0 * total_sales / sum(total_sales) OVER (), 2), '%') AS percentage_of_total
FROM category_sales
ORDER BY total_sales DESC
""")
display(df_category_share)

### Strategic Insights (Category Share)
- **Revenue Concentration**: Highlights whether a single category (such as Bikes) generates the lion's share of enterprise revenue.
- **Portfolio Diversification**: Assesses vulnerability to market disruptions in the dominant product segment.

## 2. Geographic Regional Revenue Contribution

### Business Context & Objective
Analyzes the proportional contribution of each country to overall enterprise revenue.

In [ ]:
# COMMAND ----------
# 2. Regional Sales Share Analysis
df_region_share = spark.sql("""
WITH region_sales AS (
    SELECT
        c.country AS region,  
        sum(f.sales_amount) AS total_sales
    FROM gold.fact_sales f
    LEFT JOIN gold.dim_customers c 
        ON f.customer_key = c.customer_key
    GROUP BY c.country
)
SELECT
    region,
    total_sales,
    sum(total_sales) OVER () AS overall_sales,
    concat(round(100.0 * total_sales / sum(total_sales) OVER (), 2), '%') AS percentage_of_total
FROM region_sales
ORDER BY total_sales DESC
""")
display(df_region_share)

### Business Recommendations & Summary
1. **Catalog Balance**: Strategically cross-sell accessories and apparel to bike purchasers to diversify category reliance.
2. **International Expansion**: Allocate targeted expansion initiatives toward promising secondary markets.
3. **Supply Chain Prioritization**: Direct core inventory to high-concentration territories representing over 50% of revenue.